In [1]:
import numpy as np

In [2]:
np.random.seed(42)

conv_weights = np.random.randn(8, 3, 3, 3).astype(np.float32)

# Create different ranges
conv_weights[0] *= 0.1
conv_weights[1] *= 0.5
conv_weights[6] *= 5.0
conv_weights[7] *= 10.0

print("Shape :", conv_weights.shape)

Shape : (8, 3, 3, 3)


In [3]:
def per_tensor_quantize(tensor):

    max_abs = np.max(np.abs(tensor))

    scale = max_abs / 127

    quantized = np.round(tensor / scale)

    quantized = np.clip(quantized, -127, 127)

    quantized = quantized.astype(np.int8)

    dequantized = quantized.astype(np.float32) * scale

    return quantized, dequantized, scale

In [4]:
def per_channel_quantize(tensor):

    out_channels = tensor.shape[0]

    quantized = np.zeros_like(tensor, dtype=np.int8)

    dequantized = np.zeros_like(tensor, dtype=np.float32)

    scales = np.zeros(out_channels)

    for c in range(out_channels):

        channel = tensor[c]

        max_abs = np.max(np.abs(channel))

        if max_abs == 0:
            scale = 1.0
        else:
            scale = max_abs / 127

        scales[c] = scale

        q = np.round(channel / scale)

        q = np.clip(q, -127, 127)

        q = q.astype(np.int8)

        dq = q.astype(np.float32) * scale

        quantized[c] = q

        dequantized[c] = dq

    return quantized, dequantized, scales

In [5]:
def calculate_mae(original, reconstructed):

    return np.mean(np.abs(original - reconstructed))

In [6]:
pt_q, pt_dq, pt_scale = per_tensor_quantize(conv_weights)

pc_q, pc_dq, pc_scales = per_channel_quantize(conv_weights)

In [7]:
print("="*110)

print("{:<8}{:<25}{:<18}{:<18}{:<18}{:<18}{:<10}".format(
    "Channel",
    "Range(min,max)",
    "PT Scale",
    "PT MAE",
    "PC Scale",
    "PC MAE",
    "Better"
))

print("="*110)

pt_maes = []

pc_maes = []

for c in range(8):

    original = conv_weights[c]

    pt_error = calculate_mae(original, pt_dq[c])

    pc_error = calculate_mae(original, pc_dq[c])

    pt_maes.append(pt_error)

    pc_maes.append(pc_error)

    better = "Per-Channel" if pc_error < pt_error else "Per-Tensor"

    print("{:<8}{:<25}{:<18.6f}{:<18.6f}{:<18.6f}{:<18.6f}{:<10}".format(
        c,
        f"({original.min():.2f},{original.max():.2f})",
        pt_scale,
        pt_error,
        pc_scales[c],
        pc_error,
        better
    ))

print("="*110)

print("Average PT MAE :", np.mean(pt_maes))

print("Average PC MAE :", np.mean(pc_maes))

Channel Range(min,max)           PT Scale          PT MAE            PC Scale          PC MAE            Better    
0       (-0.19,0.16)             0.303365          0.071464          0.001507          0.000326          Per-Channel
1       (-0.98,0.93)             0.303365          0.072564          0.007715          0.001661          Per-Channel
2       (-2.62,1.56)             0.303365          0.072175          0.020628          0.005373          Per-Channel
3       (-1.46,1.89)             0.303365          0.071594          0.014852          0.003731          Per-Channel
4       (-1.92,2.46)             0.303365          0.070571          0.019396          0.003995          Per-Channel
5       (-1.61,1.87)             0.303365          0.068939          0.014691          0.003901          Per-Channel
6       (-5.35,13.60)            0.303365          0.077800          0.107093          0.027714          Per-Channel
7       (-15.15,38.53)           0.303365          0.064038      